In [ ]:
import re
import hashlib
import os
import hmac as _hmac

COMMON_PASSWORDS = {
    "password", "password123", "password@123",
    "admin", "welcome",
}

def password_strength(password: str) -> dict:
    if not isinstance(password, str):
        raise TypeError(f"password must be str, got {type(password).__name__}")

    if password.lower() in COMMON_PASSWORDS:
        return {
            "score": 0,
            "level": "Very Weak",
            "suggestions": ["Avoid passwords that are very common."]
        }

    score = 0
    suggestions = []

    if len(password) >= 12:
        score += 2
    elif len(password) >= 8:
        score += 1
    else:
        suggestions.append("Use at least 12 characters.")

    if re.search(r"[A-Z]", password): score += 1
    else: suggestions.append("Add uppercase letters.")

    if re.search(r"[a-z]", password): score += 1
    else: suggestions.append("Add lowercase letters.")

    if re.search(r"[0-9]", password): score += 1
    else: suggestions.append("Add numbers.")

    if re.search(r"[^\w\s]", password): score += 1
    else: suggestions.append("Add special characters like !, @, #.")

    if score <= 2:   level = "Weak"
    elif score <= 4: level = "Medium"
    else:            level = "Strong"

    return {"level": level, "score": score, "suggestions": suggestions}


def hash_password(password: str) -> dict:
    salt = os.urandom(16)
    dk = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, iterations=600_000)
    return {"salt": salt.hex(), "hash": dk.hex()}


def verify_password(password: str, salt_hex: str, hash_hex: str) -> bool:
    salt = bytes.fromhex(salt_hex)
    dk = hashlib.pbkdf2_hmac("sha256", password.encode(), salt, iterations=600_000)
    return _hmac.compare_digest(dk.hex().encode(), hash_hex.encode())


if __name__ == "__main__":
    password = input("Enter your password: ")
    result = password_strength(password)
    stored = hash_password(password)

    print(f"\nStrength : {result['level']} ({result['score']}/6)")
    print(f"Salt     : {stored['salt']}")
    print(f"Hash     : {stored['hash']}")

    if result["suggestions"]:
        print("\nSuggestions:")
        for s in result["suggestions"]:
            print(f"  • {s}")